# MentorMatch Analytics — Full Analysis Notebook
**Companion to the Streamlit dashboard.** Same data, same feature sets, same models — runnable top-to-bottom.

Two distinct business questions (never conflated):
1. **What drives subscription conversion** (Free → Basic/Premium)
2. **What drives positive career outcomes** for mentees

> ℹ️ **Dataset note:** all 7 UAE emirates are represented — Fujairah and Umm Al Quwain rows are synthetic (bootstrap-sampled from smaller-emirate profiles, flagged transparently), since the original survey had no respondents there.

> ⚠️ **Critical data issue handled throughout:** `career_outcome` is almost perfectly separated by `income_growth_pct` and `goals_completed` — the label was derived from them. Both are **excluded** from the outcome classifier's features (target leakage); `income_growth_pct` is instead used as the **regression target**. `churned`/`days_to_subscribe` are **structurally missing** for the 706 Free users (undefined, not unobserved) — never imputed.

In [1]:
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio

from sklearn.model_selection import StratifiedKFold, KFold, cross_validate, train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import (confusion_matrix, roc_auc_score, roc_curve, accuracy_score,
                             precision_score, recall_score, f1_score, r2_score,
                             mean_squared_error, silhouette_score, adjusted_rand_score)
from scipy.cluster.hierarchy import linkage, dendrogram, fcluster
from statsmodels.stats.outliers_influence import variance_inflation_factor
from mlxtend.preprocessing import TransactionEncoder
from mlxtend.frequent_patterns import apriori, association_rules

pio.templates.default = "plotly_white"
RND = 42
pd.set_option("display.max_columns", 40)

## 1 · Data Cleaning (auditable before/after)
Fixes: gender casing, emirate whitespace, two <1% median imputations (stated, not silent).
**Deliberately not fixed:** structural NaNs in `churned`/`days_to_subscribe` — encoded, not imputed.

In [2]:
from pathlib import Path
_csv = next(p for p in [Path("MentorMatch_UAE_Clean.csv"), Path("data/MentorMatch_UAE_Clean.csv")] if p.exists())
df = pd.read_csv(_csv)
print("Shape:", df.shape)

# -- gender casing
print("\ngender BEFORE:", df["gender"].value_counts().to_dict())
df["gender"] = df["gender"].str.strip().str.title()
print("gender AFTER :", df["gender"].value_counts().to_dict())

# -- emirate whitespace
print("\nemirate BEFORE:", df["emirate"].value_counts().to_dict())
df["emirate"] = df["emirate"].str.strip()
print("emirate AFTER :", df["emirate"].value_counts().to_dict())

# -- small-gap median imputation (<1% each): justified, not silent
for col in ["monthly_income_aed", "avg_session_rating"]:
    n = int(df[col].isna().sum()); med = float(df[col].median())
    df[col] = df[col].fillna(med)
    print(f"\n{col}: imputed {n} missing ({n/len(df):.2%}) with median {med:.2f}")

# -- structural missingness: undefined for Free users, NOT unobserved
n_free = int((df["subscription_tier"] == "Free").sum())
assert df.loc[df.subscription_tier=="Free", "churned"].isna().all()
df["churned"] = df["churned"].fillna("Never Subscribed")   # explicit category
df["converted"] = (df["subscription_tier"] != "Free").astype(int)
print(f"\nStructural NaNs: {n_free} Free users -> churned='Never Subscribed'; "
      "days_to_subscribe left NaN BY DESIGN (churn analysed on paid subset only).")

# -- bands for association mining
df["age_band"] = pd.cut(df["age"], [17,24,30,40,100], labels=["18-24","25-30","31-40","40+"])
df["experience_band"] = pd.cut(df["years_experience"], [-.1,2,5,10,100],
                               labels=["0-2 yrs","3-5 yrs","6-10 yrs","10+ yrs"])
df.head(3)

Shape: (1334, 32)

gender BEFORE: {'Male': 740, 'Female': 568, 'MALE': 20, 'FEMALE': 6}
gender AFTER : {'Male': 760, 'Female': 574}

emirate BEFORE: {'Dubai': 745, 'Abu Dhabi': 310, 'Sharjah': 143, 'Ras Al Khaimah': 39, 'Fujairah': 32, 'Ajman': 29, 'Umm Al Quwain': 16, ' Dubai ': 12, ' Abu Dhabi ': 7, ' Sharjah ': 1}
emirate AFTER : {'Dubai': 757, 'Abu Dhabi': 317, 'Sharjah': 144, 'Ras Al Khaimah': 39, 'Fujairah': 32, 'Ajman': 29, 'Umm Al Quwain': 16}

monthly_income_aed: imputed 10 missing (0.75%) with median 6850.00

avg_session_rating: imputed 9 missing (0.67%) with median 3.40

Structural NaNs: 730 Free users -> churned='Never Subscribed'; days_to_subscribe left NaN BY DESIGN (churn analysed on paid subset only).


,respondent_id,age,gender,nationality_region,emirate,education_level,field_of_study,target_industry,years_experience,career_stage,employment_status,monthly_income_aed,goal_type,primary_challenge,signup_channel,subscription_tier,months_active,sessions_attended,messages_sent,num_mentors,mentor_industry_match,mentor_match_score,avg_session_rating,response_time_hours,goals_completed,income_growth_pct,skills_gained,confidence_score,referred_friend,days_to_subscribe,churned,career_outcome,converted,age_band,experience_band
0,MM0001,29,Female,African,Dubai,Bachelor's,Healthcare,Marketing & Media,2.2,Entry-Level,Unemployed,0.0,Promotion,Visa / Sponsorship,Google Search,Free,9,6,27,1,No,62,3.3,36,3,10.2,2,3.8,Yes,NaN,Never Subscribed,Limited,0,25-30,3-5 yrs
1,MM0002,23,Female,African,Dubai,Bachelor's,Finance,Hospitality & Tourism,0.7,Student / Fresh Grad,Unemployed,0.0,Career Switch,Interview Preparation,Referral,Free,6,10,20,1,Yes,53,4.0,8,3,5.2,3,3.7,No,NaN,Never Subscribed,Limited,0,18-24,0-2 yrs
2,MM0003,32,Male,South Asian,Abu Dhabi,PhD,IT / Computer Science,Consulting,3.9,Junior,Employed,11700.0,Salary Increase,Salary Negotiation,LinkedIn,Premium,6,11,40,2,No,64,3.3,1,3,8.8,5,3.6,No,30.0,No,Positive,1,31-40,3-5 yrs


## 2 · Descriptive Analytics — and the leakage evidence
The violin below is the smoking gun: `income_growth_pct` separates the label almost perfectly. This is a genuine descriptive **finding**, not a modeling footnote.

In [3]:
print(df.groupby("career_outcome")[["income_growth_pct","goals_completed","confidence_score"]].mean().round(2))
fig = px.violin(df, x="career_outcome", y="income_growth_pct", color="career_outcome",
                box=True, title="Target leakage: income_growth_pct vs career_outcome",
                color_discrete_map={"Positive":"#00C2A8","Limited":"#FF8A5C"})
fig.show()

                income_growth_pct  goals_completed  confidence_score
career_outcome                                                      
Limited                     -1.34             1.44              2.72
Positive                    20.90             3.07              5.62


In [4]:
# Cross-tabs of career_outcome against the spec'd segment variables (not income alone)
for var in ["career_stage","goal_type","primary_challenge","education_level",
            "target_industry","mentor_industry_match","subscription_tier","employment_status"]:
    ct = pd.crosstab(df[var], df["career_outcome"], normalize="index").mul(100).round(1)
    print(f"\n=== % outcome by {var} ===")
    print(ct.sort_values("Positive", ascending=False))


=== % outcome by career_stage ===
career_outcome        Limited  Positive
career_stage                           
Mid-Level                23.7      76.3
Junior                   38.8      61.2
Entry-Level              52.8      47.2
Student / Fresh Grad     52.8      47.2

=== % outcome by goal_type ===
career_outcome     Limited  Positive
goal_type                           
Salary Increase       34.5      65.5
Job Placement         36.5      63.5
Promotion             38.9      61.1
Career Switch         51.3      48.7
Networking            51.6      48.4
Skill Development     54.8      45.2

=== % outcome by primary_challenge ===
career_outcome                Limited  Positive
primary_challenge                              
Salary Negotiation               37.8      62.2
Career Direction                 42.3      57.7
Interview Preparation            44.7      55.3
Visa / Sponsorship               45.1      54.9
Limited Professional Network     46.2      53.8
Skills Gap           

In [5]:
NUMERIC = ["age","years_experience","monthly_income_aed","months_active","sessions_attended",
           "messages_sent","num_mentors","mentor_match_score","avg_session_rating",
           "response_time_hours","goals_completed","income_growth_pct","skills_gained","confidence_score"]
corr = df[NUMERIC].corr()
fig = go.Figure(go.Heatmap(z=corr.round(2).values, x=corr.columns, y=corr.columns,
                           text=corr.round(2).values, texttemplate="%{text}",
                           colorscale="RdBu", zmid=0))
fig.update_layout(title="Correlation heatmap — numeric features", height=650)
fig.show()
print("confidence_score x income_growth_pct r =",
      round(corr.loc["confidence_score","income_growth_pct"], 3),
      "-> borderline co-outcome, handled as a sensitivity analysis in §4")

confidence_score x income_growth_pct r = 0.787 -> borderline co-outcome, handled as a sensitivity analysis in §4


## 3 · Diagnostic Analytics — why outcomes differ
Stage × challenge interaction; Visa/Sponsorship by nationality (UAE sponsorship-linked employment context); mentor match as a candidate driver of **both** business goals, checked against effort.

In [6]:
pivot = df.pivot_table(index="primary_challenge", columns="career_stage",
                       values="career_outcome",
                       aggfunc=lambda s: (s=="Positive").mean()).mul(100).round(1)
print("% Positive outcome per stage x challenge cell:")
print(pivot)

visa_share = (df["primary_challenge"].eq("Visa / Sponsorship")
              .groupby(df["nationality_region"]).mean().mul(100).round(1).sort_values(ascending=False))
print("\n% citing Visa/Sponsorship as primary challenge, by nationality region:")
print(visa_share)
print("-> concentrated in sponsorship-dependent expatriate groups, consistent with the "
      "UAE's employer-sponsored residence structure (see README for cited sources).")

% Positive outcome per stage x challenge cell:
career_stage                  Entry-Level  Junior  Mid-Level  \
primary_challenge                                              
Career Direction                     53.7    56.4       75.0   
Interview Preparation                50.5    66.2       66.7   
Limited Professional Network         45.8    59.4       76.3   
Salary Negotiation                   53.1    61.5       83.3   
Skills Gap                           37.9    65.2       74.4   
Visa / Sponsorship                   45.7    60.0       85.7   

career_stage                  Student / Fresh Grad  
primary_challenge                                   
Career Direction                              55.0  
Interview Preparation                         30.8  
Limited Professional Network                  45.9  
Salary Negotiation                            60.9  
Skills Gap                                    44.4  
Visa / Sponsorship                            50.0  

% citing Visa/S

In [7]:
# Mentor match vs BOTH goals — and is it just disguised effort?
df["match_q"] = pd.qcut(df["mentor_match_score"], 4, labels=["Q1","Q2","Q3","Q4"])
print(df.groupby("match_q", observed=True).agg(
    income_growth=("income_growth_pct","mean"),
    conversion=("converted","mean"),
    positive=("career_outcome", lambda s: (s=="Positive").mean())).round(3))

df["effort_band"] = pd.qcut(df["sessions_attended"], 3, labels=["Low","Mid","High"])
print("\nIncome growth by effort band x industry-matched mentor (crude effort control):")
print(df.pivot_table(index="effort_band", columns="mentor_industry_match",
                     values="income_growth_pct", observed=True).round(2))
print("-> matched mentors sit higher at EVERY effort band: not just disguised effort.")

         income_growth  conversion  positive
match_q                                     
Q1               0.966       0.123     0.243
Q2               8.275       0.319     0.467
Q3              13.642       0.565     0.682
Q4              22.770       0.852     0.880

Income growth by effort band x industry-matched mentor (crude effort control):
mentor_industry_match     No    Yes
effort_band                        
Low                    -2.17   5.97
Mid                     4.77  13.83
High                   15.48  25.88
-> matched mentors sit higher at EVERY effort band: not just disguised effort.


## 4 · Classification — Career Outcome (leakage-free)
**Excluded:** `income_growth_pct`, `goals_completed` (leakage). `confidence_score` is borderline (r≈0.79) — run **both with and without** it to expose the sensitivity.
Four models, each with a stated role: KNN (distance baseline, expected to suffer the curse of dimensionality after one-hot), Decision Tree (interpretable but high-variance), Random Forest (variance-reduced ensemble), Gradient Boosting (often best on tabular, overfit-prone at n≈1,286 — hence judged on 5-fold CV, never a single split).

In [8]:
OUTCOME_FEATURES = ["age","years_experience","education_level","career_stage","employment_status",
                    "goal_type","primary_challenge","sessions_attended","messages_sent","num_mentors",
                    "mentor_match_score","avg_session_rating","response_time_hours","skills_gained",
                    "mentor_industry_match","months_active"]

def encode(dframe, features):
    X = pd.get_dummies(dframe[features], drop_first=True)
    return pd.DataFrame(StandardScaler().fit_transform(X), columns=X.columns, index=dframe.index)

def classifier_suite():
    return {"KNN": KNeighborsClassifier(n_neighbors=15),
            "Decision Tree": DecisionTreeClassifier(max_depth=6, random_state=RND),
            "Random Forest": RandomForestClassifier(n_estimators=300, max_depth=8,
                                                    random_state=RND, n_jobs=-1),
            "Gradient Boosting": GradientBoostingClassifier(n_estimators=200, max_depth=3,
                                                            random_state=RND)}

def evaluate(X, y, label):
    skf = StratifiedKFold(5, shuffle=True, random_state=RND)
    rows = {}
    for name, m in classifier_suite().items():
        cv = cross_validate(m, X, y, cv=skf,
                            scoring=["accuracy","precision","recall","f1","roc_auc"], n_jobs=-1)
        rows[name] = {k.replace("test_",""): f"{v.mean():.3f}±{v.std():.3f}"
                      for k, v in cv.items() if k.startswith("test_")}
    out = pd.DataFrame(rows).T
    print(f"\n=== {label} — 5-fold stratified CV ===")
    print(out)
    return out

y_out = (df["career_outcome"] == "Positive").astype(int)
print(f"Class balance: {int((y_out==0).sum())} Limited / {int((y_out==1).sum())} Positive "
      "-> mildly imbalanced; stratified CV preserves the ratio. Majority-class accuracy "
      f"is {y_out.mean():.1%}, so accuracy alone is NOT reported in isolation.")

X_no_conf = encode(df, OUTCOME_FEATURES)
res_no = evaluate(X_no_conf, y_out, "WITHOUT confidence_score (primary, conservative spec)")

X_conf = encode(df, OUTCOME_FEATURES + ["confidence_score"])
res_with = evaluate(X_conf, y_out, "WITH confidence_score (sensitivity)")
print("\nInterpretation: if AUC jumps materially when confidence_score enters, that is "
      "evidence it behaves like a CO-OUTCOME (self-reported after progress), not a leading "
      "predictor — a discussion point, not a free accuracy boost.")

Class balance: 591 Limited / 743 Positive -> mildly imbalanced; stratified CV preserves the ratio. Majority-class accuracy is 55.7%, so accuracy alone is NOT reported in isolation.



=== WITHOUT confidence_score (primary, conservative spec) — 5-fold stratified CV ===
                      accuracy    precision       recall           f1  \
KNN                0.747±0.027  0.786±0.027  0.750±0.041  0.767±0.027   
Decision Tree      0.771±0.030  0.826±0.021  0.746±0.054  0.783±0.034   
Random Forest      0.798±0.032  0.816±0.014  0.821±0.059  0.818±0.033   
Gradient Boosting  0.794±0.025  0.820±0.019  0.808±0.044  0.813±0.025   

                       roc_auc  
KNN                0.824±0.027  
Decision Tree      0.822±0.014  
Random Forest      0.892±0.023  
Gradient Boosting  0.882±0.017  



=== WITH confidence_score (sensitivity) — 5-fold stratified CV ===
                      accuracy    precision       recall           f1  \
KNN                0.798±0.027  0.828±0.025  0.805±0.038  0.816±0.026   
Decision Tree      0.845±0.014  0.862±0.021  0.861±0.054  0.860±0.017   
Random Forest      0.882±0.025  0.893±0.010  0.896±0.046  0.894±0.024   
Gradient Boosting  0.867±0.016  0.880±0.014  0.882±0.038  0.880±0.017   

                       roc_auc  
KNN                0.877±0.025  
Decision Tree      0.909±0.014  
Random Forest      0.953±0.015  
Gradient Boosting  0.948±0.014  

Interpretation: if AUC jumps materially when confidence_score enters, that is evidence it behaves like a CO-OUTCOME (self-reported after progress), not a leading predictor — a discussion point, not a free accuracy boost.


In [9]:
# Hold-out diagnostics for every model (confusion matrix + ROC + importances)
Xtr, Xte, ytr, yte = train_test_split(X_no_conf, y_out, test_size=.25, stratify=y_out, random_state=RND)
roc_fig = go.Figure(); importances = {}
for name, m in classifier_suite().items():
    m.fit(Xtr, ytr)
    pred, proba = m.predict(Xte), m.predict_proba(Xte)[:,1]
    cm = confusion_matrix(yte, pred)
    print(f"\n{name}: acc {accuracy_score(yte,pred):.3f} | prec {precision_score(yte,pred):.3f} | "
          f"rec {recall_score(yte,pred):.3f} | F1 {f1_score(yte,pred):.3f} | "
          f"AUC {roc_auc_score(yte,proba):.3f}\nconfusion matrix (rows=true L/P):\n{cm}")
    fpr, tpr, _ = roc_curve(yte, proba)
    roc_fig.add_trace(go.Scatter(x=fpr, y=tpr, name=f"{name} ({roc_auc_score(yte,proba):.3f})"))
    if hasattr(m, "feature_importances_"):
        importances[name] = pd.Series(m.feature_importances_, index=X_no_conf.columns)
roc_fig.add_trace(go.Scatter(x=[0,1], y=[0,1], line=dict(dash="dot"), name="Chance"))
roc_fig.update_layout(title="ROC — career outcome (leakage-free hold-out)", height=430)
roc_fig.show()

top_rf = importances["Random Forest"].nlargest(12)
px.bar(top_rf.sort_values(), orientation="h",
       title="Random Forest — top 12 outcome predictors (check vs §3 narrative)").show()


KNN: acc 0.766 | prec 0.790 | rec 0.790 | F1 0.790 | AUC 0.846
confusion matrix (rows=true L/P):
[[109  39]
 [ 39 147]]

Decision Tree: acc 0.790 | prec 0.854 | rec 0.753 | F1 0.800 | AUC 0.863
confusion matrix (rows=true L/P):
[[124  24]
 [ 46 140]]



Random Forest: acc 0.853 | prec 0.866 | rec 0.871 | F1 0.869 | AUC 0.920
confusion matrix (rows=true L/P):
[[123  25]
 [ 24 162]]



Gradient Boosting: acc 0.826 | prec 0.852 | rec 0.833 | F1 0.842 | AUC 0.913
confusion matrix (rows=true L/P):
[[121  27]
 [ 31 155]]


## 5 · Regression — Income Growth %
Target: `income_growth_pct` (the granular version of outcome). Same feature set as §4 minus co-outcomes.
**VIF first** — it is the actual justification for regularization, not a four-model checklist.

In [10]:
X_reg = encode(df, OUTCOME_FEATURES)   # confidence_score excluded: co-outcome
y_reg = df["income_growth_pct"]

num_cols = ["age","years_experience","sessions_attended","messages_sent","num_mentors",
            "mentor_match_score","avg_session_rating","response_time_hours",
            "skills_gained","months_active"]
Xv = X_reg[num_cols].astype(float).values
vif = pd.Series([variance_inflation_factor(Xv, i) for i in range(Xv.shape[1])],
                index=num_cols).sort_values(ascending=False)
print("VIF (scaled features):\n", vif.round(2))
print("\nReading: the engagement trio (sessions/messages/months) proxy the same latent "
      "'usage volume'. If VIF > ~5 anywhere, OLS coefficients are unstable -> regularization "
      "is MOTIVATED. If VIF is mild, regularization is insurance and OLS≈Ridge≈Lasso in CV "
      "is itself the honest finding.")

kf = KFold(5, shuffle=True, random_state=RND)
reg_models = {"OLS": LinearRegression(), "Ridge (a=1)": Ridge(alpha=1.0),
              "Lasso (a=0.1)": Lasso(alpha=0.1, max_iter=20000),
              "ElasticNet (a=0.1,l1=.5)": ElasticNet(alpha=0.1, l1_ratio=0.5, max_iter=20000)}
rows = {}
for name, m in reg_models.items():
    cv = cross_validate(m, X_reg, y_reg, cv=kf, scoring=["r2","neg_root_mean_squared_error"])
    rows[name] = {"R2 (CV)": cv["test_r2"].mean(),
                  "RMSE (CV)": -cv["test_neg_root_mean_squared_error"].mean()}
print("\n", pd.DataFrame(rows).T.round(3))

VIF (scaled features):
 sessions_attended      5.62
messages_sent          3.39
skills_gained          2.58
mentor_match_score     2.24
avg_session_rating     1.77
months_active          1.36
num_mentors            1.35
years_experience       1.09
response_time_hours    1.03
age                    1.00
dtype: float64

Reading: the engagement trio (sessions/messages/months) proxy the same latent 'usage volume'. If VIF > ~5 anywhere, OLS coefficients are unstable -> regularization is MOTIVATED. If VIF is mild, regularization is insurance and OLS≈Ridge≈Lasso in CV is itself the honest finding.



                           R2 (CV)  RMSE (CV)
OLS                         0.466     11.963
Ridge (a=1)                 0.466     11.962
Lasso (a=0.1)               0.468     11.938
ElasticNet (a=0.1,l1=.5)    0.468     11.938


In [11]:
# Shrinkage paths: watch the correlated engagement trio under Lasso
alphas = np.logspace(-3, 2, 30)
watch = ["sessions_attended","messages_sent","months_active","mentor_match_score",
         "skills_gained","mentor_industry_match_Yes"]
paths = {w: [] for w in watch}
for a in alphas:
    m = Lasso(alpha=a, max_iter=20000).fit(X_reg, y_reg)
    for w in watch:
        paths[w].append(m.coef_[list(X_reg.columns).index(w)])
fig = go.Figure()
for w in watch:
    fig.add_trace(go.Scatter(x=alphas, y=paths[w], name=w))
fig.update_layout(title="Lasso shrinkage paths — redundant engagement twins zero out first",
                  xaxis_type="log", xaxis_title="lambda", yaxis_title="coefficient", height=430)
fig.show()

## 6 · Classification — Subscription Conversion
**Different target, different rationale** — not §4 reused. `referred_friend`/`signup_channel` enter (conversion-specific); `churned`/`days_to_subscribe` excluded (exist only *after* conversion — circular). `mentor_match_score` appears in **both** models: that overlap is itself a finding.

In [12]:
CONV_FEATURES = ["signup_channel","referred_friend","mentor_match_score","sessions_attended",
                 "messages_sent","avg_session_rating","primary_challenge","goal_type",
                 "target_industry","career_stage","response_time_hours"]
y_conv = df["converted"]
print(f"Free {int((y_conv==0).sum())} vs Converted {int((y_conv==1).sum())}")
X_conv = encode(df, CONV_FEATURES)
res_conv = evaluate(X_conv, y_conv, "Conversion (Free vs Basic/Premium)")

rf = RandomForestClassifier(n_estimators=300, max_depth=8, random_state=RND, n_jobs=-1).fit(X_conv, y_conv)
px.bar(pd.Series(rf.feature_importances_, index=X_conv.columns).nlargest(12).sort_values(),
       orientation="h", title="RF — top conversion drivers (note mentor_match_score in BOTH models)").show()

# Churn: defined ONLY for the 580 paid users — subset analysis, never imputed
paid = df[df["converted"] == 1]
print(f"\nPaid subset n={len(paid)} | churn rate {(paid['churned']=='Yes').mean():.1%} | "
      f"median days_to_subscribe {paid['days_to_subscribe'].median():.0f}")
print(paid.assign(ch=paid['churned'].eq('Yes')).groupby('ch')
      [["sessions_attended","messages_sent","mentor_match_score","avg_session_rating"]]
      .mean().round(2).rename(index={False:"Retained", True:"Churned"}))

Free 730 vs Converted 604



=== Conversion (Free vs Basic/Premium) — 5-fold stratified CV ===
                      accuracy    precision       recall           f1  \
KNN                0.766±0.024  0.819±0.038  0.621±0.035  0.706±0.031   
Decision Tree      0.804±0.026  0.808±0.040  0.745±0.037  0.774±0.030   
Random Forest      0.845±0.019  0.854±0.025  0.793±0.028  0.822±0.022   
Gradient Boosting  0.850±0.018  0.848±0.025  0.816±0.034  0.831±0.022   

                       roc_auc  
KNN                0.837±0.018  
Decision Tree      0.853±0.041  
Random Forest      0.921±0.017  
Gradient Boosting  0.921±0.020  



Paid subset n=604 | churn rate 15.2% | median days_to_subscribe 7
          sessions_attended  messages_sent  mentor_match_score  \
ch                                                               
Retained              12.22          49.88               70.57   
Churned               11.59          44.01               71.32   

          avg_session_rating  
ch                            
Retained                3.51  
Churned                 3.55  


## 7 · Clustering — Engagement Segments
Behavioral features only; demographics excluded (one-hot dummies distort Euclidean distance — a demographic segmentation would need k-modes/Gower, named as a limitation). StandardScaler mandatory (`months_active` 0–24 vs `mentor_match_score` up to 99). K chosen from **elbow + silhouette together**; hierarchical clustering validates.

In [13]:
CLUSTER_FEATURES = ["months_active","sessions_attended","messages_sent","num_mentors",
                    "mentor_match_score","avg_session_rating","goals_completed",
                    "skills_gained","confidence_score","response_time_hours"]
Xs = StandardScaler().fit_transform(df[CLUSTER_FEATURES])

ks = range(2, 16); inertia, sil = [], []
for k in ks:
    km = KMeans(n_clusters=k, n_init=10, random_state=RND).fit(Xs)
    inertia.append(km.inertia_); sil.append(silhouette_score(Xs, km.labels_))
best_k = list(ks)[int(np.argmax(sil))]
fig = go.Figure()
fig.add_trace(go.Scatter(x=list(ks), y=inertia, name="Inertia (elbow)", yaxis="y"))
fig.add_trace(go.Scatter(x=list(ks), y=sil, name="Silhouette", yaxis="y2"))
fig.update_layout(title=f"Elbow vs silhouette — silhouette peaks at K={best_k}; "
                        "elbow flattens gradually (ambiguous alone) -> choose from BOTH",
                  yaxis=dict(title="inertia"), yaxis2=dict(title="silhouette",
                  overlaying="y", side="right"), height=420)
fig.show()

km = KMeans(n_clusters=best_k, n_init=10, random_state=RND).fit(Xs)
df["cluster"] = km.labels_
print(f"Chosen K={best_k} | silhouette={silhouette_score(Xs, km.labels_):.3f}")

Chosen K=3 | silhouette=0.306


In [14]:
# PCA -> 3D (raw 10-D can't be plotted directly; report fidelity of the projection)
pca = PCA(n_components=3, random_state=RND)
P = pca.fit_transform(Xs)
print("Variance explained by 3 PCs:", pca.explained_variance_ratio_.round(3),
      "| total:", round(pca.explained_variance_ratio_.sum(), 3),
      "-> the plot is a faithful sketch, not the full geometry.")
p3 = pd.DataFrame(P, columns=["PC1","PC2","PC3"]); p3["cluster"] = df["cluster"].astype(str)
px.scatter_3d(p3, x="PC1", y="PC2", z="PC3", color="cluster", opacity=.7,
              title=f"K-Means (K={best_k}) in PCA space").update_traces(marker_size=3).show()

print("\nSegment profiles (means):")
print(df.groupby("cluster")[CLUSTER_FEATURES].mean().round(2))
print("\nBusiness view per segment:")
print(df.groupby("cluster").agg(n=("respondent_id","count"), conversion=("converted","mean"),
      positive=("career_outcome", lambda s: (s=="Positive").mean()),
      income_growth=("income_growth_pct","mean")).round(3))

Variance explained by 3 PCs: [0.476 0.126 0.101] | total: 0.703 -> the plot is a faithful sketch, not the full geometry.



Segment profiles (means):
         months_active  sessions_attended  messages_sent  num_mentors  \
cluster                                                                 
0                10.37              14.15          58.06         1.92   
1                 4.26               0.00           3.64         0.00   
2                 6.69               5.96          23.11         1.19   

         mentor_match_score  avg_session_rating  goals_completed  \
cluster                                                            
0                     70.82                3.54             3.92   
1                      0.00                0.00             0.26   
2                     58.42                3.39             1.56   

         skills_gained  confidence_score  response_time_hours  
cluster                                                        
0                 5.26              6.10                11.14  
1                 0.46              1.49                15.16  
2         

In [15]:
# Validation: Ward hierarchical clustering on the same features
rng = np.random.RandomState(RND)
idx = rng.choice(len(Xs), 400, replace=False)
Z = linkage(Xs[idx], method="ward")
for kk in range(2, 7):
    lh = fcluster(Z, t=kk, criterion="maxclust")
    lk = KMeans(n_clusters=kk, n_init=10, random_state=RND).fit(Xs[idx]).labels_
    print(f"K={kk}: hierarchical silhouette {silhouette_score(Xs[idx], lh):.3f} | "
          f"ARI vs K-Means {adjusted_rand_score(lh, lk):.3f}")
print("\nIf the hierarchical silhouette peaks near K-Means' choice and ARI is well above 0, "
      "the segments are unlikely to be an artifact of K-Means' spherical assumption.")

K=2: hierarchical silhouette 0.244 | ARI vs K-Means 0.704
K=3: hierarchical silhouette 0.264 | ARI vs K-Means 0.676
K=4: hierarchical silhouette 0.197 | ARI vs K-Means 0.654
K=5: hierarchical silhouette 0.167 | ARI vs K-Means 0.451


K=6: hierarchical silhouette 0.163 | ARI vs K-Means 0.587

If the hierarchical silhouette peaks near K-Means' choice and ARI is well above 0, the segments are unlikely to be an artifact of K-Means' spherical assumption.


## 8 · Association Rule Mining
**Apriori over FP-Growth, justified:** ~13 categorical attributes and 1,286 transactions make Apriori's candidate-generation cost trivial, and its levelwise rules read naturally for a business audience; FP-Growth's tree compression only pays off at much larger scale.

In [16]:
ITEM_COLS = ["goal_type","primary_challenge","career_stage","education_level","target_industry",
             "mentor_industry_match","subscription_tier","career_outcome","employment_status",
             "signup_channel","referred_friend","age_band","experience_band"]
tx = df[ITEM_COLS].astype(str).apply(lambda r: [f"{c}={r[c]}" for c in ITEM_COLS], axis=1).tolist()
te = TransactionEncoder()
basket = pd.DataFrame(te.fit(tx).transform(tx), columns=te.columns_)
freq = apriori(basket, min_support=0.05, use_colnames=True, max_len=3)
rules = association_rules(freq, metric="confidence", min_threshold=0.6)
rules["antecedents"] = rules["antecedents"].apply(lambda s: ", ".join(sorted(s)))
rules["consequents"] = rules["consequents"].apply(lambda s: ", ".join(sorted(s)))
print(f"{len(freq)} frequent itemsets (support>=0.05), {len(rules)} rules (confidence>=0.6)")

oc = rules[rules["consequents"].str.startswith("career_outcome=")]
print("\nTop rules landing on career_outcome, by lift:")
print(oc.nlargest(10, "lift")[["antecedents","consequents","support","confidence","lift"]]
      .round(3).to_string(index=False))

1638 frequent itemsets (support>=0.05), 1081 rules (confidence>=0.6)

Top rules landing on career_outcome, by lift:
                                          antecedents                                         consequents  support  confidence  lift
                               career_stage=Mid-Level   career_outcome=Positive, experience_band=6-10 yrs    0.088       0.661 5.039
                            subscription_tier=Premium career_outcome=Positive, employment_status=Employed    0.133       0.774 2.211
                            subscription_tier=Premium  career_outcome=Positive, mentor_industry_match=Yes    0.142       0.822 1.937
     mentor_industry_match=No, subscription_tier=Free                              career_outcome=Limited    0.237       0.823 1.857
    career_stage=Mid-Level, subscription_tier=Premium                             career_outcome=Positive    0.056       1.000 1.795
  experience_band=6-10 yrs, subscription_tier=Premium                             care

## 9 · Findings & Prescriptive Recommendations

**What predicts mentee success (post-leakage-correction):** engagement volume, `skills_gained`, and mentor match quality — with honest, moderate accuracy (the near-perfect scores a leaky model would show are gone, by design).

**What predicts conversion:** `mentor_match_score`, early engagement, `referred_friend`, `signup_channel`.

**The overlap — the headline:** `mentor_match_score` ranks in both models. Improving matching serves revenue *and* mentee value at once.

**Prescriptive actions:** (1) proactive mentor re-matching for the bottom match-score quartile; (2) formal referral incentives; (3) channel re-weighting toward high-converting signup channels; (4) a dedicated Visa/Sponsorship playbook for the 206 visa-challenged mentees, concentrated in sponsorship-dependent nationality groups; (5) industry match as a default matching constraint; (6) early-engagement onboarding nudges for the low-engagement cluster; (7) churn watchlist for paid users whose engagement drops below segment median.

**Limitations, stated plainly:** observational data — a mentor match *predicting* income growth is correlational and may reflect selection (motivated mentees both secure better matches and progress faster); `career_outcome` is a constructed label; `confidence_score` is plausibly a co-outcome (sensitivity shown in §4); churn is unobservable for Free users; the data is a cross-sectional snapshot. The honest next step for any recommended lever is an A/B test.